# Step 1: Connect to GPU

Connect the T4 machine on top right

# Step 2: Install Required Libraries
Install necessary libraries such as PyTorch, segmentation_models_pytorch, and other dependencies using pip.

In [ ]:
 # Install Required Libraries
!pip install -q torch torchvision segmentation-models-pytorch matplotlib pillow ipywidgets tifffile scikit-image

# Step 4: Load the Pretrained Model
Load the saved model weights using `torch.load` and initialize the model architecture.


## Step 4a: download the FAST model
The model can be downloaded with request access at https://drive.google.com/file/d/1SKEUXr1xOf739tIzdnSMx03lGNaAWY84/view?usp=sharing

## Step 4b: Place the downloaded model (.pth file) in this directory

In [1]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

MODEL_IN_CHANNELS = 1
model = smp.UnetPlusPlus(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=MODEL_IN_CHANNELS,
    classes=5  # Number of classes in the segmentation task
).to(device)

# Load the saved model weights downloaded from google drive link above
model_path = "best_model.pth"

if not os.path.exists(model_path):
    raise FileNotFoundError(
        f"Model weights not found at: {model_path}\n"
        "Set `model_path` to a valid .pth file in this workspace."
    )

state = torch.load(model_path, map_location=device)
model.load_state_dict(state)
model.eval()
print(f"Loaded weights: {model_path}")

Loaded weights: best_model.pth


# Step5: Interactive preprocessing + inference

## Step 5a: Upload a sample image from your dataset that you want to test

## Step 5b: Provide the path to your file in the box or in the code

## Step 5c: Preform pre processing that gives the best predicted mask with FAST
Use the sliders to tune preprocessing (including rolling-ball background subtraction). When you're happy, click **Run inference** to see the segmentation result on your image.

In [2]:
import os
import cv2
import numpy as np
import torch
import tifffile as tiff
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
from skimage.restoration import rolling_ball
import ipywidgets as widgets
from IPython.display import display, clear_output

transform = transforms.Compose([
    transforms.ToTensor(),
])


# Make this cell runnable even if the preview cell wasn't executed
if "DEFAULT_IMAGE_PATH" not in globals():
    DEFAULT_IMAGE_PATH = "test.tif"

def read_grayscale_image(path: str) -> np.ndarray:
    if path.lower().endswith(('.tif', '.tiff')):
        img = tiff.imread(path)
    else:
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    return img

def preprocess_image_np(
    img: np.ndarray,
    bg_radius: int = 50,
    subtract_background: bool = True,
) -> np.ndarray:
    
    if subtract_background and bg_radius > 0:
        bg = rolling_ball(img, radius=int(bg_radius))
        img = img - bg
    return img

def to_model_tensor(img_2d: np.ndarray) -> torch.Tensor:
    img_np = Image.fromarray(img_2d).convert("L")
    img = transform(img_np).unsqueeze(0).to(device)
    return img

def decode_segmentation_masks(mask: np.ndarray, colormap: np.ndarray) -> np.ndarray:
    out_rgb = np.zeros((mask.shape[0], mask.shape[1], 3), dtype=np.uint8)
    for cls in range(colormap.shape[0]):
        out_rgb[mask == cls] = colormap[cls]
    return out_rgb

# Class colors (adjust if you have a preferred palette)
COLORMAP = np.array([
    [0, 0, 0],       # 0 Background
    [0, 255, 0],     # 1 Actin
    [255, 0, 0],     # 2 Focal Adhesions
    [255, 255, 0],   # 3 Lamellipodia
    [0, 0, 255],     # 4 Filopodia
], dtype=np.uint8)

# Ensure model is in eval mode once loaded
try:
    model.eval()
except NameError:
    raise RuntimeError("Model is not defined yet. Run the model-loading cell first.")

# Widgets
path_w = widgets.Text(value=DEFAULT_IMAGE_PATH, description="Image path", layout=widgets.Layout(width="80%"))
bg_on_w = widgets.Checkbox(value=True, description="Subtract background")
bg_radius_w = widgets.IntSlider(value=50, min=0, max=200, step=1, description="BG radius")
run_btn = widgets.Button(description="Run inference", button_style="primary")

status_w = widgets.HTML(value="<b>Status:</b> Ready")
progress_w = widgets.IntProgress(value=0, min=0, max=4, description="Progress")
progress_w.layout = widgets.Layout(width="60%")

preview_out = widgets.Output()
results_out = widgets.Output()

def show_input_preview(*_):
    with preview_out:
        clear_output(wait=True)
        p = path_w.value.strip()
        if not p:
            print("Enter an image path to preview.")
            return
        if not os.path.exists(p):
            print(f"File not found: {p}")
            return
        try:
            raw = read_grayscale_image(p)
        except Exception as e:
            print(f"Failed to load image: {e}")
            return
        plt.figure(figsize=(6, 6))
        plt.imshow(raw, cmap="gray")
        plt.title(f"Input preview: {os.path.basename(p)}\nshape={raw.shape}, dtype={raw.dtype}")
        plt.axis("off")
        plt.show()

def set_status(text: str, *, bar_style: str = "") -> None:
    status_w.value = f"<b>Status:</b> {text}"
    progress_w.bar_style = bar_style

def _run_inference(_):
    run_btn.disabled = True
    progress_w.value = 0
    set_status("Loading image…", bar_style="info")
    try:
        img_path = path_w.value.strip()
        if not img_path:
            raise ValueError("Please provide an image path.")
        if not os.path.exists(img_path):
            raise FileNotFoundError(f"File not found: {img_path}")

        raw = read_grayscale_image(img_path)
        # Always show the input image first (even before preprocessing)
        show_input_preview()
        progress_w.value = 1

        set_status("Preprocessing…", bar_style="info")
        corrected = preprocess_image_np(
            raw,
            bg_radius=int(bg_radius_w.value),
            subtract_background=bool(bg_on_w.value),
        )
        progress_w.value = 2

        set_status("Running model…", bar_style="info")
        img_t = to_model_tensor(corrected)
        with torch.no_grad():
            logits = model(img_t)
            pred = torch.argmax(logits, dim=1).squeeze(0).detach().cpu().numpy().astype(np.int32)
        progress_w.value = 3

        set_status("Rendering…", bar_style="info")
        rgb = decode_segmentation_masks(pred, COLORMAP)
        with results_out:
            clear_output(wait=True)
            _, axes = plt.subplots(1, 3, figsize=(18, 6))
            axes[0].imshow(raw, cmap="gray")
            axes[0].set_title("Original")
            axes[0].axis("off")
            axes[1].imshow(corrected, cmap="gray")
            axes[1].set_title("Preprocessed")
            axes[1].axis("off")
            axes[2].imshow(rgb)
            axes[2].set_title("Prediction")
            axes[2].axis("off")
            plt.tight_layout()
            plt.show()
        progress_w.value = 4
        set_status("Done", bar_style="success")
    except Exception as e:
        set_status(f"Error: {e}", bar_style="danger")
        with results_out:
            clear_output(wait=True)
            print(e)
    finally:
        run_btn.disabled = False

run_btn.on_click(_run_inference)
path_w.observe(show_input_preview, names="value")

# Initial preview (shows whatever is currently in the textbox)
show_input_preview()

display(widgets.VBox([
    path_w,
    preview_out,
    widgets.HBox([bg_on_w, bg_radius_w]),
    widgets.HBox([run_btn, progress_w, status_w]),
    results_out,
]))

Loaded weights: best_model.pth
